In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict, load_dataset
from tqdm import tqdm
import random, re
import os

## Generate CHILDES datasets splits 

In [ ]:
# AMOUNT OF DATA for each split
TRAIN_TOKENS = 10_000_000
DEV_TOKENS = 2_510_000
TEST_TOKENS = 2_510_000
SEED = 42


# --- PREPROCESSING FUNCTION ---
def clean_examples(examples):
    cleaned_texts = []
    for sent in examples["text"]:

        if sent.startswith("%add:"):
            continue

        # skip child utterances
        if sent.startswith("*CHI"):
            continue

        # skip lines within brackets like: [xxx]
        if sent.startswith("[") and sent.endswith("]"):
            continue

        # delete all text between [...] and also remove trailing .
        sent = re.sub(r"\[.*?\](\.?)", "", sent)

        # skip metadata lines like:
        # = = = childes/childes_na/brown/eve/010800.cha = = =
        if re.match(r"^=+.*=+$", sent.strip()):
            continue

        # delete empty lines
        if sent.strip() == "":
            continue

        # keep only text after first tab
        if "\t" in sent:
            sent = sent.split("\t", 1)[1]

        cleaned_texts.append(sent.lower().strip())

    return {"text": cleaned_texts}


def count_tokens(line):
    return len(token_pattern.findall(line))

def count_total_tokens(sentences):
    return sum(count_tokens(s) for s in sentences)


# --- LOAD DATASET ---
raw_dataset = load_dataset(
    "text", 
    data_files={"train": "./corpora/source/train_100M/childes.train"}
)["train"]


# --- APPLY the preprocessing function ---
cleaned_dataset = raw_dataset.map(
    clean_examples,
    batched=True,
    remove_columns=["text"]
)

# count tokens using regex
token_pattern = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")


# Build dataset with token count 
all_sentences = cleaned_dataset["text"]
dataset = [{"line": s, "tokens": count_tokens(s)} for s in tqdm(all_sentences, desc="Counting tokens")]

# Sampler function 
def sample_sentences_fast(dataset, tokens_needed, seed=SEED):
    random.seed(seed)
    indices = list(range(len(dataset)))
    random.shuffle(indices)
    sampled = []
    total_tokens = 0
    for idx in indices:
        if total_tokens + dataset[idx]["tokens"] > tokens_needed:
            break
        sampled.append(idx)
        total_tokens += dataset[idx]["tokens"]
    return sampled


## Actual Split

# Step 1: Dev
dev_indices = sample_sentences_fast(dataset, DEV_TOKENS)
remaining_indices = list(set(range(len(dataset))) - set(dev_indices))

# Step 2: Test
test_dataset = [dataset[i] for i in remaining_indices]
test_indices_rel = sample_sentences_fast(test_dataset, TEST_TOKENS)
test_indices = [remaining_indices[i] for i in test_indices_rel]

# Step 3: Train
train_candidates = list(set(range(len(dataset))) - set(dev_indices) - set(test_indices))
train_subset = [dataset[i] for i in train_candidates]
train_indices_rel = sample_sentences_fast(train_subset, TRAIN_TOKENS)
train_indices = [train_candidates[i] for i in train_indices_rel]

# Order
train_indices.sort()
dev_indices.sort()
test_indices.sort()

# Sentences
train_sentences = [dataset[i]["line"] for i in train_indices]
dev_sentences = [dataset[i]["line"] for i in dev_indices]
test_sentences = [dataset[i]["line"] for i in test_indices]


print(f"Train: {len(train_sentences)} sentences, ~{count_total_tokens(train_sentences):,} tokens")
print(f"Dev:   {len(dev_sentences)} sentences, ~{count_total_tokens(dev_sentences):,} tokens")
print(f"Test:  {len(test_sentences)} sentences, ~{count_total_tokens(test_sentences):,} tokens")

# ===========================
# SAVE OUTPUT
# ===========================
base = "./corpora/CHILDES_rand/original"

with open(f"{base}/childes.train.txt", "w", encoding="utf-8") as f:
    for s in train_sentences:
        f.write(s + "\n")

with open(f"{base}/childes.dev.txt", "w", encoding="utf-8") as f:
    for s in dev_sentences:
        f.write(s + "\n")

with open(f"{base}/childes.test.txt", "w", encoding="utf-8") as f:
    for s in test_sentences:
        f.write(s + "\n")



Counting tokens: 100%|██████████| 3505881/3505881 [00:52<00:00, 66508.85it/s]


Train: 2318378 sentences, ~9,999,998 tokens
Dev:   581777 sentences, ~2,510,000 tokens
Test:  582557 sentences, ~2,509,999 tokens


## Generate WIKIPEDIA datasets splits 

In [ ]:
wiki_files = [
    './corpora/source/wikipedia1.txt',
    './corpora/source/wikipedia2.txt'
]

In [ ]:
# AMOUNT OF DATA for each split
TRAIN_TOKENS = 10_000_000
DEV_TOKENS = 2_510_000
TEST_TOKENS = 2_510_000
TOTAL_TOKENS = TRAIN_TOKENS + DEV_TOKENS + TEST_TOKENS

# --- cleaning and token count patterns with regex ---
pattern = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")
paren_pattern = re.compile(r"\([^)]*\)")

def clean_sentence(sentence):
    sentence = paren_pattern.sub("", sentence.lower())
    return sentence.strip()

def count_tokens(sentence):
    return len(pattern.findall(sentence))

def count_total_tokens(sentences):
    return sum(count_tokens(s) for s in sentences)

space_before_punct = re.compile(r"\s+([.,!?;:])")

def fix_spaces(line):
    return space_before_punct.sub(r"\1", line)


# --- load all sentences ---
all_sentences = []
for path in wiki_files:
    with open(path, 'r', encoding='utf-8') as f:
        all_sentences.extend(f.readlines())



dataset = [{"line": clean_sentence(s), "tokens": count_tokens(clean_sentence(s))}
           for s in tqdm(all_sentences, desc="Cleaning and counting tokens")]


# === STEP 1: CAP TO TOTAL TOKENS NEEDED ===
print(f"Selecting ~{TOTAL_TOKENS:,} tokens total...")
selected_indices = sample_sentences_fast(dataset, TOTAL_TOKENS)
subset = [dataset[i] for i in selected_indices]

# === STEP 2: SAMPLE DEV ===
dev_indices_rel = sample_sentences_fast(subset, DEV_TOKENS)
dev_indices = [selected_indices[i] for i in dev_indices_rel]

# === STEP 3: SAMPLE TEST (from remaining) ===
remaining_for_test = list(set(selected_indices) - set(dev_indices))
test_subset = [dataset[i] for i in remaining_for_test]
test_indices_rel = sample_sentences_fast(test_subset, TEST_TOKENS)
test_indices = [remaining_for_test[i] for i in test_indices_rel]

# === STEP 4: TRAIN = REMAINING ===
train_indices = list(set(selected_indices) - set(dev_indices) - set(test_indices))


train_indices.sort()
dev_indices.sort()
test_indices.sort()

train_sentences = [dataset[i]["line"] for i in train_indices if dataset[i]["line"].strip()]
dev_sentences = [dataset[i]["line"] for i in dev_indices if dataset[i]["line"].strip()]
test_sentences = [dataset[i]["line"] for i in test_indices if dataset[i]["line"].strip()]


print(f"Train: {len(train_sentences)} sentences, ~{count_total_tokens(train_sentences):,} tokens")
print(f"Dev:   {len(dev_sentences)} sentences, ~{count_total_tokens(dev_sentences):,} tokens")
print(f"Test:  {len(test_sentences)} sentences, ~{count_total_tokens(test_sentences):,} tokens")

# --- CREATE HF DATASETS ---
train_dataset = Dataset.from_dict({"text": train_sentences})
dev_dataset = Dataset.from_dict({"text": dev_sentences})
test_dataset = Dataset.from_dict({"text": test_sentences})

dataset_dict = DatasetDict({
    "train": train_dataset,
    "validation": dev_dataset,
    "test": test_dataset
})

# --- SAVE SPLITS ---
with open("./corpora/WIKI_rand/original/wiki.train.txt", "w", encoding="utf-8") as f:
    for line in dataset_dict["train"]["text"]:
        if line.strip():
            f.write(fix_spaces(line.strip()) + '\n')

with open("./corpora/WIKI_rand/original/wiki.dev.txt", "w", encoding="utf-8") as f:
    for line in dataset_dict["validation"]["text"]:
        if line.strip():
            f.write(fix_spaces(line.strip()) + '\n')

with open("./corpora/WIKI_rand/original/wiki.test.txt", "w", encoding="utf-8") as f:
    for line in dataset_dict["test"]["text"]:
        if line.strip():
            f.write(fix_spaces(line.strip()) + '\n')

## Process CANDOR dataset (we received it by email upon request)


You access the folder that the authors of Candor share with you (in my case is in my Download folder) and we process it in order to collect all the text files in a unique .txt called *candor_dataset.txt*.

In [ ]:
main_folder = './Downloads/no_media'
output_file = './corpora/source/candor_dataset.txt'

# Collect all utterances
utterances = []

for root, dirs, files in os.walk(main_folder):
    for dir in dirs:
        trans_path = os.path.join(root,dir, "transcription", "transcript_cliffhanger.csv")
        if os.path.exists(trans_path):
            try:
                df = pd.read_csv(trans_path)
                if "utterance" in df.columns:
                    utterances.extend(df["utterance"].dropna().astype(str).tolist())
                else:
                    print(f"[WARNING] 'utterance' column not found in {trans_path}")
            except Exception as e:
                print(f"[ERROR] Could not read {trans_path}: {e}")

# Write to final file
with open(output_file, "w", encoding="utf-8") as f:
    for utt in utterances:
        f.write(utt.strip() + "\n")

print(f"Saved {len(utterances)} utterances to {output_file}")


✅ Saved 0 utterances to /Users/frapadovani/Desktop/syntactic-bootstrapping/corpora/source/candor_dataset.txt


## Generate CANDOR datasets splits 
In order to generate the file *bnc_spoken_to_be_used_for_candor.txt* please scroll down in this Notebook until you reach the section **FULL METADATA ANALYSIS**. 
We conducted an analysis of the type of sources used for the spoken BNC portion in order to select the most appropriate type of data for this data domain. 

#### Start with preprocessing and cleaning Switchboard

In [ ]:
input_file = "./corpora/source/train_100M/switchboard.train"
output_file = "./corpora/source/train_100M/switchboard.train.cleaned.txt"

# Pattern to match speaker tags at the beginning of a line
speaker_pattern = re.compile(r"^[AB]:\s*")

with open(input_file, "r", encoding="utf-8") as fin, open(output_file, "w", encoding="utf-8") as fout:
    for line in fin:
        # Remove speaker tags
        cleaned_line = speaker_pattern.sub("", line).strip()
        if cleaned_line:  # skip empty lines
            fout.write(cleaned_line + "\n")

print(f"Saved cleaned file to: {output_file}")

In [ ]:
# input files 
bnc_spoken_filtered = "./corpora/source/bnc_spoken_to_be_used_for_candor.txt"
switchboard = "./source/train_100M/switchboard.train.cleaned.txt"
candor = "./corpora/source/candor_dataset.txt"

# Output files
dev_file = "./corpora/CANDOR_rand/original/candor.dev.txt"
test_file = "./corpora/CANDOR_rand/original/candor.test.txt"
train_file = "./corpora/CANDOR_rand/original/candor.train.txt"

In [ ]:
pattern = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")

In [ ]:
def count_tokens(line):
    return len(pattern.findall(line))

def tokenize_count(line):
    return len(pattern.findall(line))

def load_dataset(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]
    return [{"line": line, "tokens": tokenize_count(line)} for line in lines]

In [ ]:
def sample_sentences_fast_v2(dataset, tokens_needed):
    indices = list(range(len(dataset)))
    random.shuffle(indices)

    sampled = []
    total_tokens = 0

    for idx in indices:
        if total_tokens >= tokens_needed:
            break
        sampled.append(idx)
        total_tokens += dataset[idx]["tokens"]

    return sampled


In [87]:
bnc_spoken_filtered_data = load_dataset(bnc_spoken_filtered)
switchboard_data = load_dataset(switchboard)
candor_data = load_dataset(candor)

datasets = {
    "bnc_spoken_filtered": bnc_spoken_filtered_data,
    "switchboard": switchboard_data,
    "candor": candor_data
}

In [88]:
# Count total tokens in each dataset
dataset_tokens = {name: sum(line["tokens"] for line in lines) for name, lines in datasets.items()}
total_tokens = sum(dataset_tokens.values())

In [89]:
dataset_proportions = {name: tokens / total_tokens for name, tokens in dataset_tokens.items()}
dataset_proportions

{'bnc_spoken_filtered': 0.35919410380569966,
 'switchboard': 0.07359978016195803,
 'candor': 0.5672061160323423}

In [ ]:
dev_target = 2_510_000
test_target = 2_510_000
train_target = 10_000_000

dataset_proportions = {
    "bnc_spoken_filtered": len(bnc_spoken_filtered_data),
    "switchboard": len(switchboard_data),
    "candor": len(candor_data)
}
total_length = sum(dataset_proportions.values())
dataset_proportions = {k: v / total_length for k, v in dataset_proportions.items()}

dev_indices = {}
test_indices = {}

for dataset_name, data in datasets.items():
    proportion = dataset_proportions[dataset_name]
    dev_tokens_needed = int(dev_target * proportion)
    test_tokens_needed = int(test_target * proportion)

    print(f"Sampling for {dataset_name}: dev={dev_tokens_needed} tokens, test={test_tokens_needed} tokens")

    dev_sample = sample_sentences_fast_v2(data, dev_tokens_needed)
    remaining_indices = [i for i in range(len(data)) if i not in dev_sample]
    remaining_data = [data[i] for i in remaining_indices]

    test_sample_local = sample_sentences_fast_v2(remaining_data, test_tokens_needed)
    test_sample = [remaining_indices[i] for i in test_sample_local]

    dev_indices[dataset_name] = set(dev_sample)
    test_indices[dataset_name] = set(test_sample)

# Create dev/test/train sets
dev_set, test_set, train_set = [], [], []

for dataset_name, lines in datasets.items():
    for idx, s in enumerate(lines):
        if idx in dev_indices[dataset_name]:
            dev_set.append(s["line"])
        elif idx in test_indices[dataset_name]:
            test_set.append(s["line"])
        else:
            train_set.append(s["line"])

# Ensure training set token count is ~10M
train_tokens = sum(count_tokens(line) for line in train_set)
print(f"Training tokens before trimming: {train_tokens}")

if train_tokens > 10_000_000:
    trimmed_train = []
    tokens_count = 0
    for line in train_set:
        if tokens_count >= 10_000_000:
            break
        trimmed_train.append(line)
        tokens_count += count_tokens(line)
    train_set = trimmed_train
    train_tokens = tokens_count

print(f"Final training tokens: {train_tokens}")

# Save sets
print("Saving dev, test, and train sets...")
with open(dev_file, "w", encoding="utf-8") as f:
    f.write("\n".join(dev_set))

with open(test_file, "w", encoding="utf-8") as f:
    f.write("\n".join(test_set))

with open(train_file, "w", encoding="utf-8") as f:
    f.write("\n".join(train_set))

print("Dataset creation complete.")

## Generate BNC (WRITTEN) datasets splits 

In [ ]:
from lxml import etree as ET
import os
import string
import codecs
import csv
import re

# Input and output directories
root_dir = "2554"
output_dir = "./corpora/source/bnc_written"
metadata_file = './corpora/source/bnc_written_metadata.csv'

os.makedirs(output_dir, exist_ok=True)

pattern = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")

# Token budgets
train_budget = 10_000_000
dev_budget = 2_510_000
test_budget = 2_510_000
TOTAL_TOKENS = train_budget + dev_budget + test_budget

# All sentences collected
all_sentences = []
sentence_token_counts = []
used_metadata = []

def write_file(uttr):
    return "".join(uttr)

def process_xml(xml_file):
    try:
        parser = ET.XMLParser(recover=True)
        tree = ET.parse(xml_file, parser)
        root = tree.getroot()
        if root is None:
            return None, False, []
    except ET.XMLSyntaxError as e:
        print(f"Skipping {xml_file}: {e}")
        return None, False, []

    text_class = root.find(".//textClass/catRef")
    if text_class is None or "WRI" not in text_class.attrib.get("targets", ""):
        return None, False, []

    title = root.findtext(".//titleStmt/title", default="unknown").strip()
    publisher = root.findtext(".//sourceDesc/bibl/imprint/publisher", default="unknown").strip()
    pubPlace = root.findtext(".//sourceDesc/bibl/imprint/pubPlace", default="unknown").strip()
    creation_date = root.findtext(".//profileDesc/creation", default="unknown").strip()

    sentences = []
    for child in root[1].iter():
        if child.tag == "s":
            uttr = []
            for token in child.iter():
                if token.tag in ("w", "c") and token.text:
                    uttr.append(token.text)
            if uttr:
                sentence = write_file(uttr)
                n_tokens = len(pattern.findall(sentence))
                if not any(c in string.ascii_letters + string.digits for c in sentence):
                    continue
                sentences.append((sentence, n_tokens))

    if sentences:
        return {
            "xml_file": os.path.basename(xml_file),
            "title": title,
            "publisher": publisher,
            "pubPlace": pubPlace,
            "creation_date": creation_date
        }, True, sentences

    return None, False, []

# Sampler function that returns indices
def sample_sentences_fast(token_counts, tokens_needed):
    indices = list(range(len(token_counts)))
    random.shuffle(indices)
    sampled = []
    total_tokens = 0
    for idx in indices:
        if total_tokens >= tokens_needed:
            break
        sampled.append(idx)
        total_tokens += token_counts[idx]
    return sampled

# Step 1 — Process XML files
total_tokens = 0
for dirpath, dirnames, files in os.walk(root_dir):
    for f in files:
        if f.endswith(".xml") and total_tokens < TOTAL_TOKENS:
            xml_path = os.path.join(dirpath, f)
            meta, used, sentences = process_xml(xml_path)
            if used and meta:
                used_metadata.append(meta)
                for sent, n_tokens in sentences:
                    all_sentences.append(sent)
                    sentence_token_counts.append(n_tokens)
                    total_tokens += n_tokens
                    if total_tokens >= TOTAL_TOKENS:
                        break
        if total_tokens >= TOTAL_TOKENS:
            break

print(f"Collected {len(all_sentences)} sentences (~{total_tokens} tokens)")

# Step 2 — Sample dev, test, train
random.seed(42)

dev_indices = sample_sentences_fast(sentence_token_counts, dev_budget)
remaining_indices = list(set(range(len(all_sentences))) - set(dev_indices))

remaining_token_counts = [sentence_token_counts[i] for i in remaining_indices]
test_indices_rel = sample_sentences_fast(remaining_token_counts, test_budget)
test_indices = [remaining_indices[i] for i in test_indices_rel]

train_indices = list(set(range(len(all_sentences))) - set(dev_indices) - set(test_indices))

# Step 3 — Preserve order
train_indices.sort()
dev_indices.sort()
test_indices.sort()

train_sentences = [all_sentences[i] for i in train_indices if all_sentences[i].strip()]
dev_sentences = [all_sentences[i] for i in dev_indices if all_sentences[i].strip()]
test_sentences = [all_sentences[i] for i in test_indices if all_sentences[i].strip()]

def count_total_tokens(sentences):
    return sum(len(pattern.findall(s)) for s in sentences)

print(f"Train: {len(train_sentences)} sentences, ~{count_total_tokens(train_sentences):,} tokens")
print(f"Dev:   {len(dev_sentences)} sentences, ~{count_total_tokens(dev_sentences):,} tokens")
print(f"Test:  {len(test_sentences)} sentences, ~{count_total_tokens(test_sentences):,} tokens")

# Step 4 — Save splits
for split_name, sentences in zip(["train", "dev", "test"], [train_sentences, dev_sentences, test_sentences]):
    out_path = os.path.join(output_dir, f"bnc.{split_name}.txt")
    with codecs.open(out_path, "w", "utf-8") as f:
        for sent in sentences:
            f.write(sent + "\n")
    print(f"{split_name} saved: {out_path} ({count_total_tokens(sentences)} tokens)")

# Step 5 — Write metadata
with open(metadata_file, "w", newline="", encoding="utf-8") as csvfile:
    fieldnames = ["xml_file", "title", "publisher", "pubPlace", "creation_date"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(used_metadata)

print(f"Metadata written for {len(used_metadata)} files.")

# FULL METADATA ANALYSIS (checking both Spoken and Written BNC metadata)

In [ ]:
# Input folder
root_dir = "2554"

# Output CSVs
metadata_wri = "./corpora/source/bnc_metadata_WRI.csv"
metadata_spo = "./corpora/source/bnc_metadata_SPO.csv"

# Regex pattern for words
pattern = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")

# Token counters
total_tokens = {"WRI": 0, "SPO": 0}

def process_xml(xml_file):
    """Parse XML and extract metadata + token count"""
    try:
        parser = ET.XMLParser(recover=True)
        tree = ET.parse(xml_file, parser)
        root = tree.getroot()
        if root is None:
            print(f"No root element in {xml_file}")
            return None
    except ET.XMLSyntaxError as e:
        print(f"Skipping {xml_file}: {e}")
        return None
    except Exception as e:
        print(f"Error parsing {xml_file}: {e}")
        return None

    # text class (WRI or SPO, sometimes others)
    text_class_tag = root.find(".//textClass/catRef")
    if text_class_tag is None:
        print(f"No <textClass><catRef> found in {xml_file}")
        return None

    text_class = text_class_tag.attrib.get("targets", "UNKNOWN")

    # Extract metadata
    title = root.findtext(".//titleStmt/title", default="unknown").strip()
    publisher = root.findtext(".//sourceDesc/bibl/imprint/publisher", default="unknown").strip()
    pubPlace = root.findtext(".//sourceDesc/bibl/imprint/pubPlace", default="unknown").strip()
    creation_date = root.findtext(".//profileDesc/creation", default="unknown").strip()

    # Count tokens in all <s> (sentences)
    token_count = 0
    for child in root.iter():
        if child.tag == "s":
            uttr = []
            for token in child.iter():
                if token.tag in ("w", "c") and token.text:
                    uttr.append(token.text)
            if uttr:
                sentence = "".join(uttr)
                n_tokens = len(pattern.findall(sentence))
                if any(c in string.ascii_letters + string.digits for c in sentence):
                    token_count += n_tokens

    return {
        "xml_file": os.path.basename(xml_file),
        "text_class": text_class,
        "title": title,
        "publisher": publisher,
        "pubPlace": pubPlace,
        "creation_date": creation_date,
        "tokens": token_count
    }

# Prepare two writers (one for WRI, one for SPO)
fieldnames = ["xml_file", "text_class", "title", "publisher", "pubPlace", "creation_date", "tokens"]

with open(metadata_wri, "w", newline="", encoding="utf-8") as wri_csv, \
     open(metadata_spo, "w", newline="", encoding="utf-8") as spo_csv:

    wri_writer = csv.DictWriter(wri_csv, fieldnames=fieldnames)
    spo_writer = csv.DictWriter(spo_csv, fieldnames=fieldnames)

    wri_writer.writeheader()
    spo_writer.writeheader()

    # Walk over files
    for dirpath, _, files in os.walk(root_dir):
        for f in files:
            if f.endswith(".xml"):
                xml_path = os.path.join(dirpath, f)
                meta = process_xml(xml_path)
                if meta:
                    if "WRI" in meta["text_class"]:
                        wri_writer.writerow(meta)
                        total_tokens["WRI"] += meta["tokens"]
                    elif "SPO" in meta["text_class"]:
                        spo_writer.writerow(meta)
                        total_tokens["SPO"] += meta["tokens"]

print("Metadata extraction done")
print(f"Total tokens WRI: {total_tokens['WRI']}")
print(f"Total tokens SPO: {total_tokens['SPO']}")


## Compute Statistics From Metadata File

In [ ]:
metadata_wri = pd.read_csv('./bnc_written_metadata.csv')

# Ensure columns are stripped of whitespace
df = metadata_wri.applymap(lambda x: x.strip() if isinstance(x, str) else x)

/tmp/ipykernel_2533338/2925418175.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


In [ ]:
def extract_doc_type(title):
    # Common patterns
    patterns = {
        'book': r'\bbook\b',
        'television news': r'\bmaterial written to be spoken\b',
        'miscellanea': r'\bmiscellanea\b',
        'periodical': r'\bperiodical\b',
    }
    for key, pat in patterns.items():
        if re.search(pat, title, re.IGNORECASE):
            return key
    return 'other'

df['doc_type'] = df['title'].apply(extract_doc_type)

In [ ]:
df['doc_type'].unique()

array(['book', 'miscellanea', 'periodical', 'television news'],
      dtype=object)

In [ ]:
def extract_domain(title):
    match = re.search(r'\(domain:\s*([^)]+)\)', title, re.IGNORECASE)
    return match.group(1).strip() if match else 'unknown'

df['domain'] = df['title'].apply(extract_domain)

In [ ]:
df['domain']

0           social science
1           social science
2            world affairs
3              imaginative
4       belief and thought
               ...        
3136                  arts
3137               leisure
3138        social science
3139        social science
3140        social science
Name: domain, Length: 3141, dtype: object

In [ ]:
publisher_counts = df['publisher'].value_counts()
print(publisher_counts.head(20))

publisher
u.p.                            567
Oxford University Press         196
Longman Group UK Ltd            164
Newspaper Publishing plc        145
Guardian Newspapers Ltd         102
The Daily Telegraph plc          93
Routledge & Kegan Paul plc       58
APT Data Services Ltd.           57
Mills & Boon                     49
Environment Publishers Ltd       30
Headline Book Publishing plc     29
Hodder & Stoughton Ltd           27
Blackwell                        27
IPC Magazines Ltd                27
Fontana Press                    26
Penguin Group                    26
Macmillan Publishers Ltd         26
Cambridge University Press       26
Faber & Faber Ltd                25
Corgi Books                      22
Name: count, dtype: int64


In [ ]:
doc_type_counts = df['doc_type'].value_counts()
print(doc_type_counts)

doc_type
book               1415
periodical         1209
miscellanea         482
television news      35
Name: count, dtype: int64


In [ ]:
domain_counts = df['domain'].value_counts()
print(domain_counts)

domain
social science        527
world affairs         482
imaginative           475
leisure               437
applied science       370
commerce              295
arts                  262
natural sciences      146
belief and thought    145
unknown                 1
world news              1
Name: count, dtype: int64


## Analyse metadata for Spoken BNC 

In [ ]:
metadata_spo = "./corpora/source/bnc_metadata_SPO.csv"

In [23]:
# Load your dataset
df = pd.read_csv(metadata_spo)

# Extract first sentence from title
df["first_sentence"] = df["title"].astype(str).str.split(".").str[0].str.strip()

# Compute frequency of first sentences
freq = df["first_sentence"].value_counts().reset_index()
freq.columns = ["first_sentence", "count"]

# Save to CSV
freq.to_csv("title_first_sentence_frequency.csv", index=False, encoding="utf-8")

print(freq[:10])  # Display rows 10 to 20

                                      first_sentence  count
0                               Medical consultation     72
1                              Medical consultations     32
2                    Oral history project: interview     19
3                              Albert Gunter: sermon     15
4  General practitioner's surgery: medical consul...     15
5         Nottingham Oral History Project: interview     14
6                        Trade Union Annual Congress      9
7                   Suffolk Sound Archive: interview      8
8    Nottinghamshire Oral History Project: interview      7
9                                          Interview      6


In [24]:
## get a list of the files that contain three specific words in the title

# Load dataset
df = pd.read_csv(metadata_spo)

# Extract first sentence from title
df["first_sentence"] = df["title"].astype(str).str.split(".").str[0].str.strip()

# Filter rows where title contains consultation(s) or conversation (case insensitive)
keywords = ["consultation", "consultations", "conversation", "meeting", "Oral history project: interview"]
mask = df["title"].astype(str).str.contains(
    "|".join(keywords), case=False, na=False
)

# Save file names (assuming your column is named 'file_name')
matching_files = df.loc[mask, "xml_file"].tolist()

print("Matching file names:", matching_files)
print(len(matching_files), "files found.")

Matching file names: ['G60.xml', 'GYK.xml', 'GYJ.xml', 'GYH.xml', 'GYM.xml', 'GYL.xml', 'GYY.xml', 'GY8.xml', 'GYN.xml', 'GY9.xml', 'GYU.xml', 'GYB.xml', 'GYC.xml', 'GY5.xml', 'GYT.xml', 'GY7.xml', 'GYA.xml', 'GY6.xml', 'GYW.xml', 'GYD.xml', 'GYS.xml', 'GYE.xml', 'GYG.xml', 'GYF.xml', 'G5B.xml', 'G54.xml', 'G5U.xml', 'G5T.xml', 'G55.xml', 'G5C.xml', 'G57.xml', 'G5V.xml', 'G5W.xml', 'G56.xml', 'G52.xml', 'G5S.xml', 'G5D.xml', 'G5E.xml', 'G5R.xml', 'G53.xml', 'G51.xml', 'G5P.xml', 'G5G.xml', 'G5F.xml', 'G50.xml', 'G5K.xml', 'G5M.xml', 'G5L.xml', 'G5N.xml', 'G58.xml', 'G5Y.xml', 'G5X.xml', 'G59.xml', 'G4P.xml', 'G4E.xml', 'G43.xml', 'G4R.xml', 'G4S.xml', 'G42.xml', 'G4D.xml', 'G46.xml', 'G4A.xml', 'G47.xml', 'G45.xml', 'G4T.xml', 'G4C.xml', 'G4B.xml', 'G44.xml', 'G49.xml', 'G4X.xml', 'G4N.xml', 'G4Y.xml', 'G48.xml', 'G3U.xml', 'FXL.xml', 'FXM.xml', 'FX9.xml', 'FXX.xml', 'FXY.xml', 'FX8.xml', 'FXN.xml', 'FXJ.xml', 'FXK.xml', 'FXH.xml', 'FXR.xml', 'FXE.xml', 'FXD.xml', 'FXF.xml', 'FXG.xml',

In [ ]:
metadata_spo = "./corpora/source/bnc_metadata_SPO.csv"
root_dir = "./2554/download/Texts"
spoken_output = "./corpora/bnc_spoken_to_be_used_for_candor.txt"
keywords = ["consultation", "consultations", "conversation", "meeting"]
pattern = re.compile(r"[A-Za-z0-9]+(?:'[A-Za-z0-9]+)?")


# === FUNCTIONS ===
def get_matching_files(metadata_csv, keywords):
    df = pd.read_csv(metadata_csv)
    df["first_sentence"] = df["title"].astype(str).str.split(".").str[0].str.strip()

    mask = df["title"].astype(str).str.contains(
        "|".join(keywords), case=False, na=False
    )

    matching_files = df.loc[mask, "xml_file"].tolist()
    print(f"🔍 {len(matching_files)} matching files found in metadata.")
    return matching_files


def find_file_in_subdirs(root_dir, target_filename):
    """Recursively search for file in root_dir."""
    for dirpath, _, files in os.walk(root_dir):
        if target_filename in files:
            return os.path.join(dirpath, target_filename)
    return None


def process_xml(xml_file, spoken_writer):
    try:
        parser = ET.XMLParser(recover=True)
        tree = ET.parse(xml_file, parser)
        root = tree.getroot()
    except ET.XMLSyntaxError as e:
        print(f"Skipping {xml_file}: {e}")
        return 0

    text_class_tag = root.find(".//textClass/catRef")
    if text_class_tag is None or "SPO" not in text_class_tag.attrib.get("targets", ""):
        return 0

    tokens_written = 0
    for child in root[1].iter():
        if child.tag == "s":
            uttr = []
            for token in child.iter():
                if token.tag in ("w", "c") and token.text:
                    uttr.append(token.text)
            if uttr:
                sentence = "".join(uttr)
                n_tokens = len(pattern.findall(sentence))
                if any(c in string.ascii_letters + string.digits for c in sentence):
                    spoken_writer.write(sentence + "\n")
                    tokens_written += n_tokens
    return tokens_written


# === MAIN PROCESS ===
if __name__ == "__main__":
    matching_files = get_matching_files(metadata_spo, keywords)
    total_tokens_spo = 0

    with open(spoken_output, "w", encoding="utf-8") as spoken_file:
        for file_rel_path in matching_files:
            file_name = os.path.basename(file_rel_path)
            xml_path = find_file_in_subdirs(root_dir, file_name)

            if xml_path:
                total_tokens_spo += process_xml(xml_path, spoken_file)
            else:
                print(f"File not found: {file_name}")

    print("Filtered dataset created")
    print(f"Total SPO tokens written: {total_tokens_spo}")
